# Dicionário de Dados e Referências

**Dicionário de Dados de Notificação Geral - SINAN:** _http://portalsinan.saude.gov.br/images/documentos/Agravos/NINDIV/DIC_DADOS_NET_Not_Individual_rev.pdf_  
**Dicionario Ficha de Notificação Individual Geral - SINAN:** _http://portalsinan.saude.gov.br/images/documentos/Agravos/DRT%20Transtorno%20Mental/DIC_DADOS_DRT_TranstornosMentais_v5.pdf_  
**Classificação Brasileira de Ocupação:** _https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/cbo/servicos/downloads/livro-1-portal-cbo.pdf_  
**Visualizador Web - TabNet:** _http://tabnet.datasus.gov.br/cgi/tabcgi.exe?sinannet/cnv/transmentalbr.def_

_A base representa os casos notificados formalmente ao sistema de vigilância nacional, sabendo que o volume real de indivíduos com Burnout tratados por psicólogos e psiquiatras particulares/de convênio médico está altamente sub-representado nesses números._

# Instalar as dependências para o R:

In [ ]:
# 1. Instalar dependências do sistema Linux necessárias para manipulação de dados e FTP
system("apt-get update && apt-get install -y libssl-dev libcurl4-openssl-dev libxml2-dev r-base-dev build-essential")

# 2. Instalar pacotes do R

options(repos = c(CRAN = "https://cloud.r-project.org"))

install.packages("remotes")
install.packages("rlang")
install.packages("tidyverse") # Inclui dplyr, ggplot2, etc.

# 3. Instalar o pacote microdatasus direto do repositório da Fiocruz (vai utilizar o CRAN setado anteriormente)
remotes::install_github("rfsaldanha/microdatasus", type = "source")

# 4. Carregar as bibliotecas
library("remotes")
library(microdatasus)
library(dplyr)

# As bibliotecas instaladas ficam em: /opt/conda/lib/R/library

Carregar/Verificar as bibliotecas:

In [8]:
library("remotes")
library(microdatasus)
library(dplyr)
#.libPaths()

# Baixando os dados referentes às bases SINAN para os anos de 2010 à 2025, com excessão de 2020 e 2021 (ausentes no SINAN).:

In [ ]:
anos_antigos <- "10 11 12 13 14 15 16 17 18 19"
anos_recentes <- "22 23 24 25"

system(paste("for ano in", anos_antigos, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/FINAIS/MENTBR$ano.dbc; done"))
system(paste("for ano in", anos_recentes, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR$ano.dbc; done"))

#system("wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR24.dbc")

library(read.dbc)
dados <- read.dbc("~/work/MENTBR25.dbc")

head(dados)

### Lendo e concatenando todos pacotes baixados referentes a cada ano:

In [ ]:
#library(read.dbc)
#library(dplyr)

# Definir os anos desejados (2010 a 2025, removendo 2020 e 2021)
anos <- 2010:2025
anos_filtrados <- anos[!anos %in% c(2020, 2021)]

# Lista vazia para armazenar os dataframes temporariamente
lista_dados <- list()

for (ano in anos_filtrados) {

  # Extrai os dois últimos dígitos do ano (ex: 2015 vira "15")
  sufixo <- substr(as.character(ano), 3, 4)

  # Monta o caminho do arquivo dinamicamente
  caminho_arquivo <- paste0("~/work/MENTBR", sufixo, ".dbc")

  # Cria um nome de identificação para a lista (ex: "ano_15")
  nome_lista <- paste0("ano_", sufixo)

  # Mensagem no console para acompanhamento do progresso da leitura
  cat("Lendo arquivo:", caminho_arquivo, "\n")

  # Lê o arquivo e armazena na lista, se existir
  if (file.exists(caminho_arquivo)) {
    lista_dados[[nome_lista]] <- read.dbc(caminho_arquivo)
  } else {
    warning(paste("Arquivo não encontrado:", caminho_arquivo))
  }
}

# Junta todos os anos em um único grande dataframe
dados_completos <- bind_rows(lista_dados, .id = "FONTE_ANO")

# Remove a lista temporária para liberar memória RAM
rm(lista_dados)

# Salvar na memória secundária
saveRDS(dados_completos, file = "work/dados_completos_sinan.rds")

# dados <- readRDS("work/dados_completos_sinan.rds")

In [12]:
dados <- readRDS("~/work/dados_completos_sinan.rds")

In [13]:
head(dados)

,FONTE_ANO,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,⋯,ANO_NASC,NU_LOTE_I,DT_TRANSUS,DT_TRANSDM,DT_TRANSSM,DT_TRANSRM,DT_TRANSRS,DT_TRANSSE,NU_LOTE_V,NU_LOTE_H
,<chr>,<fct>,<fct>,<date>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<fct>,<fct>,<date>,<date>,<date>,<date>,<date>,<date>,<fct>,<fct>
1,ano_10,2,F99,2010-01-27,201004,2010,28,280030,2056,5841399,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,ano_10,2,F99,2010-01-29,201004,2010,42,420910,1565,2436418,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,ano_10,2,F99,2010-01-05,201001,2010,35,355030,1331,2786664,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,ano_10,2,F99,2010-01-26,201004,2010,29,293330,1398,2550202,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
5,ano_10,2,F99,2010-01-15,201002,2010,29,292740,1380,2557894,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,ano_10,2,F99,2010-01-15,201002,2010,29,291480,1386,3432890,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [ ]:
# Explorando a base de dados coadunada:
names(dados)
length(dados)
dim(dados)

### **Averiguando variedade das categorias de transtornos mentais para a variável de Diagnóstico Específico:**

In [ ]:
dados$DIAG_ESP

### **Averiguando integridade e disponibilidade dos estados para a variável 'Estado de Notificação':**

In [ ]:
dados$SG_UF_NOT

In [1]:
# Vetor nomeado para servir de "dicionário" de tradução
tradutor_uf <- c(
  "11" = "RO", "12" = "AC", "13" = "AM", "14" = "RR", "15" = "PA", "16" = "AP", "17" = "TO",
  "21" = "MA", "22" = "PI", "23" = "CE", "24" = "RN", "25" = "PB", "26" = "PE", "27" = "AL",
  "28" = "SE", "29" = "BA", "31" = "MG", "32" = "ES", "33" = "RJ", "35" = "SP", "41" = "PR",
  "42" = "SC", "43" = "RS", "50" = "MS", "51" = "MT", "52" = "GO", "53" = "DF"
)

In [32]:
tabela_Burnout <- dados %>%
  mutate(
    UF_Codigo = trimws(as.character(SG_UF)),
    UF_SG = tradutor_uf[UF_Codigo],
  ) %>%
  filter(
    DIAG_ESP   == "Z730"
  ) %>%
  group_by(UF_SG) %>%
  summarise(Casos = n(), .groups = "drop") %>%
  arrange(UF_SG)

head(tabela_Burnout, n = Inf)

UF_SG,Casos
<chr>,<int>
AC,4
AL,8
AM,2
BA,238
CE,64
DF,6
ES,6
GO,4
MA,7


### Podemos verificar que não há registros referentes nem à Piauí (**PI**) nem à  Amapá (**AP**) para a base de dados utilizada.

# Selecionar as variáveis de interesse e tratar a variável de 'Estado De Notificação':

In [37]:
tabela_Burnout <- dados %>%
  select(FONTE_ANO, DIAG_ESP, SG_UF, SIT_TRAB, ID_OCUPA_N, CNAE, UF_EMP,
  TERCEIRIZA, NUTEMPORIS, TPTEMPO, AFAST_DESG, AFAST_TRAB, CAPES, EVOLUCAO) %>%
  mutate(
    UF_Codigo = trimws(as.character(SG_UF)),
    UF_SG = tradutor_uf[UF_Codigo],
  ) %>% select(-SG_UF, -UF_Codigo) %>%
    relocate(UF_SG, .after = DIAG_ESP) %>%
  filter(
    DIAG_ESP   == "Z730"
  )
# Exibe o resultado final com os dados tratados mediante dicionário:
#print(tabela_Burnout, n = Inf)

#saveRDS(tabela_Burnout, file = "work/dados_tratados_sinan.rds")
#tabela_Burnout <- readRDS('work/dados_tratados_sinan.rds')

head(tabela_Burnout)

,FONTE_ANO,DIAG_ESP,UF_SG,SIT_TRAB,ID_OCUPA_N,CNAE,UF_EMP,TERCEIRIZA,NUTEMPORIS,TPTEMPO,AFAST_DESG,AFAST_TRAB,CAPES,EVOLUCAO
,<chr>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
1,ano_10,Z730,SP,01,513425,NA,35,2,5,3,1,1,1,3
2,ano_10,Z730,PR,01,239405,NA,41,2,4,NA,2,1,1,3
3,ano_11,Z730,MT,04,411010,NA,51,2,20,4,1,NA,1,NA
4,ano_11,Z730,RO,04,223405,NA,11,2,5,4,2,2,1,3
5,ano_11,Z730,BA,07,410105,NA,NA,2,0,NA,1,1,1,3
6,ano_11,Z730,PE,04,331105,05126,26,2,20,4,1,NA,1,4


# Tratando a variável de 'Situação No Mercado De Trabalho':

In [38]:
# library(dplyr)
library(forcats)

SIT_TRAB_TRATADO <- tabela_Burnout %>%
  mutate(
    # Recodificando os níveis do fator existente
    SIT_TRAB_TRATADA = fct_recode(SIT_TRAB,
      "Empregado CLT"               = "01",
      "Empregado sem Carteira"      = "02",
      "Autônomo / Conta Própria"    = "03",
      "Trabalhador Temporário"      = "04",
      "Empregador"                  = "05",
      "Trabalhador Não Remunerado"  = "06",
      "Desempregado"                = "07",
      "Aposentado"                  = "08",
      "Ignorado"                    = "09"
    )
  )
  SIT_TRAB_TRATADO <- SIT_TRAB_TRATADO %>% relocate(SIT_TRAB_TRATADA, .after = SIT_TRAB)
  head(SIT_TRAB_TRATADO)

  #SIT_TRAB_TRATADO <- SIT_TRAB_TRATADO %>% select(-SIT_TRAB)
  #SIT_TRAB_TRATADO <- SIT_TRAB_TRATADO %>% rename(SIT_TRAB = SIT_TRAB_TRATADO)
  tabela_Burnout <- SIT_TRAB_TRATADO

,FONTE_ANO,DIAG_ESP,UF_SG,SIT_TRAB,SIT_TRAB_TRATADA,ID_OCUPA_N,CNAE,UF_EMP,TERCEIRIZA,NUTEMPORIS,TPTEMPO,AFAST_DESG,AFAST_TRAB,CAPES,EVOLUCAO
,<chr>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
1,ano_10,Z730,SP,01,Empregado CLT,513425,NA,35,2,5,3,1,1,1,3
2,ano_10,Z730,PR,01,Empregado CLT,239405,NA,41,2,4,NA,2,1,1,3
3,ano_11,Z730,MT,04,Trabalhador Temporário,411010,NA,51,2,20,4,1,NA,1,NA
4,ano_11,Z730,RO,04,Trabalhador Temporário,223405,NA,11,2,5,4,2,2,1,3
5,ano_11,Z730,BA,07,Desempregado,410105,NA,NA,2,0,NA,1,1,1,3
6,ano_11,Z730,PE,04,Trabalhador Temporário,331105,05126,26,2,20,4,1,NA,1,4


# Tratando a variável de 'Ocupação':

In [41]:
# library(dplyr)
# library(forcats)

ID_OCUPA_N_TRADADO <- tabela_Burnout %>%
  mutate(
    # Garante uma versão em texto limpa (sem alterar a ID_OCUPA_N original)
    CBO_TEXTO = trimws(as.character(ID_OCUPA_N)),
    
    # Extrai os 2 primeiros dígitos (Grande Grupo)
    CBO_GRUPO_DIGITO = substr(CBO_TEXTO, 1, 2),
    
    # Traduz os Grandes Grupos com base na tabela da CBO
    CBO_GRANDE_GRUPO = case_when(
      grepl("^0", CBO_GRUPO_DIGITO) ~ "Militares, Policiais e Bombeiros",
      grepl("^1", CBO_GRUPO_DIGITO) ~ "Diretores e Gerentes",
      grepl("^2", CBO_GRUPO_DIGITO) ~ "Profissionais das Ciências e Artes (Nível Superior)",
      grepl("^3", CBO_GRUPO_DIGITO) ~ "Técnicos de Nível Médio",
      grepl("^4", CBO_GRUPO_DIGITO) ~ "Serviços Administrativos / Escritório",
      grepl("^5", CBO_GRUPO_DIGITO) ~ "Comércio, Serviços de Atendimento e Vendas",
      grepl("^6", CBO_GRUPO_DIGITO) ~ "Agropecuária, Florestal e Pesca",
      grepl("^[789]", CBO_GRUPO_DIGITO) ~ "Produção Industrial e Construção Civil",
      TRUE ~ "Não Informado / Ignorado"
    ),
    
    # Transforma o resultado final em fator para otimizar gráficos
    CBO_GRANDE_GRUPO = as.factor(CBO_GRANDE_GRUPO)
  )
  ID_OCUPA_N_TRADADO <- ID_OCUPA_N_TRADADO %>% relocate(ID_OCUPA_N, .before = CBO_TEXTO)
  # ID_OCUPA_N_TRADADO <- ID_OCUPA_N_TRADADO %>% select(-CBO_TEXTO, -CBO_GRUPO_DIGITO, -ID_OCUPA_N)
  tabela_Burnout <- ID_OCUPA_N_TRADADO
  head(tabela_Burnout)
  # saveRDS(tabela_Burnout, file = 'work/dados_tratados_sinan.rds')

,FONTE_ANO,DIAG_ESP,UF_SG,SIT_TRAB,SIT_TRAB_TRATADA,CNAE,UF_EMP,TERCEIRIZA,NUTEMPORIS,TPTEMPO,AFAST_DESG,AFAST_TRAB,CAPES,EVOLUCAO,ID_OCUPA_N,CBO_TEXTO,CBO_GRUPO_DIGITO,CBO_GRANDE_GRUPO
,<chr>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<chr>,<fct>
1,ano_10,Z730,SP,01,Empregado CLT,NA,35,2,5,3,1,1,1,3,513425,513425,51,"Comércio, Serviços de Atendimento e Vendas"
2,ano_10,Z730,PR,01,Empregado CLT,NA,41,2,4,NA,2,1,1,3,239405,239405,23,Profissionais das Ciências e Artes (Nível Superior)
3,ano_11,Z730,MT,04,Trabalhador Temporário,NA,51,2,20,4,1,NA,1,NA,411010,411010,41,Serviços Administrativos / Escritório
4,ano_11,Z730,RO,04,Trabalhador Temporário,NA,11,2,5,4,2,2,1,3,223405,223405,22,Profissionais das Ciências e Artes (Nível Superior)
5,ano_11,Z730,BA,07,Desempregado,NA,NA,2,0,NA,1,1,1,3,410105,410105,41,Serviços Administrativos / Escritório
6,ano_11,Z730,PE,04,Trabalhador Temporário,05126,26,2,20,4,1,NA,1,4,331105,331105,33,Técnicos de Nível Médio
